# Controlador multisemilla para Senior Exp6

Esta notebook corre la workflow base `z635_WorkFlow_01_senior_exp6.ipynb` una vez por cada semilla.
En cada corrida se setean:
- `PARAM$semilla_primigenia` con el valor de la lista `semillas`
- `PARAM$experimento` con un entero incremental en 1
- `PARAM$CN$ratio`
- `PARAM$CN$desvios`
- `PARAM$boruta$aplicacion_boruta`

In [ ]:
CONFIG <- list(
  semillas = c(109843, 109847, 109849, 109859, 109873),
  nro_experimento_inicial = 1L,
  experimento_state_file = "./exp6_multisemilla/ultimo_experimento.txt",
  ratio_canaritos = 0.2,
  desvios_canaritos = 2,
  aplicacion_boruta_flag = FALSE,
  notebook_base = "z635_WorkFlow_01_senior_exp6.ipynb",
  output_dir = "./exp6_multisemilla"
)

In [ ]:
leer_notebook <- function(path) {
  nb <- jsonlite::fromJSON(path, simplifyVector = FALSE)
  nb$cells
}

requerir_parametro <- function(nombre) {
  if (!exists("PARAM", mode = "list", inherits = FALSE)) {
    stop("Falta establecer PARAM en la notebook controladora. Debes definirlo antes de correr la base.", call. = FALSE)
  }
  if (is.null(PARAM[[nombre]])) {
    stop(paste0("Falta establecer PARAM$", nombre, " en la notebook controladora."), call. = FALSE)
  }
  invisible(PARAM[[nombre]])
}

requerir_subparametro <- function(grupo, nombre) {
  if (!exists("PARAM", mode = "list", inherits = FALSE)) {
    stop("Falta establecer PARAM en la notebook controladora. Debe definirse antes de correr la base.", call. = FALSE)
  }
  if (is.null(PARAM[[grupo]]) || is.null(PARAM[[grupo]][[nombre]])) {
    stop(paste0("Falta establecer PARAM$", grupo, "$", nombre, " en la notebook controladora."), call. = FALSE)
  }
  invisible(PARAM[[grupo]][[nombre]])
}

verificar_parametros <- function() {
  requerir_parametro("semilla_primigenia")
  requerir_parametro("experimento")
  requerir_subparametro("CN", "ratio")
  requerir_subparametro("CN", "desvios")
  requerir_subparametro("boruta", "aplicacion_boruta")
  TRUE
}

reemplazar_si_existe <- function(src, patron, reemplazo) {
  if (grepl(patron, src, fixed = TRUE)) {
    gsub(patron, reemplazo, src, fixed = TRUE)
  } else {
    src
  }
}

ejecutar_corrida <- function(semilla, experimento, ratio, desvios, boruta_flag, base_path, work_dir) {
  dir.create(work_dir, recursive = TRUE, showWarnings = FALSE)

  PARAM <<- list(
    semilla_primigenia = semilla,
    experimento = experimento,
    CN = list(ratio = ratio, desvios = desvios),
    boruta = list(aplicacion_boruta = boruta_flag)
  )

  verificar_parametros()

  cat("\n[Inicio corrida] semilla=", semilla, "| experimento=", experimento, "| ratio=", ratio, "| desvios=", desvios, "| boruta=", boruta_flag, "\n")

  cells <- leer_notebook(base_path)
  for (i in seq_along(cells)) {
    cell <- cells[[i]]
    if (is.null(cell$cell_type) || !identical(cell$cell_type, "code")) next

    src <- paste(unlist(cell$source), collapse = "")

    src <- reemplazar_si_existe(src, "PARAM$semilla_primigenia <- 109843", paste0("PARAM$semilla_primigenia <- ", semilla))
    src <- reemplazar_si_existe(src, "PARAM$experimento <- 950000", paste0("PARAM$experimento <- ", experimento))
    src <- reemplazar_si_existe(src, "PARAM$CN$ratio <- 0.2", paste0("PARAM$CN$ratio <- ", ratio))
    src <- reemplazar_si_existe(src, "PARAM$CN$desvios <- 2", paste0("PARAM$CN$desvios <- ", desvios))
    src <- reemplazar_si_existe(src, "PARAM$boruta$aplicacion_boruta <- FALSE", paste0("PARAM$boruta$aplicacion_boruta <- ", if (boruta_flag) "TRUE" else "FALSE"))

    if (i == 1L) {
      src <- paste0("setwd(", shQuote(work_dir), ")\n", src)
    }

    cat("\n[Celda ", i, " / ", length(cells), "] ejecutando...\n")
    flush.console()
    eval(parse(text = src), envir = .GlobalEnv)
    cat("[Celda ", i, " / ", length(cells), "] OK\n")
    flush.console()
  }

  cat("\n[Fin corrida] semilla=", semilla, "| experimento=", experimento, "\n")
  invisible(NULL)
}

In [ ]:
base_path <- normalizePath(file.path(getwd(), CONFIG$notebook_base), winslash = "/")
if (!file.exists(base_path)) stop("No existe la notebook base: ", base_path)

leer_ultimo_experimento <- function(path) {
  if (!file.exists(path)) {
    return(NULL)
  }
  valor <- suppressWarnings(as.integer(readLines(path, warn = FALSE)[1]))
  if (is.na(valor)) {
    stop("El archivo de estado del experimento existe pero no tiene un número válido: ", path, call. = FALSE)
  }
  valor
}

guardar_ultimo_experimento <- function(path, valor) {
  dir.create(dirname(path), recursive = TRUE, showWarnings = FALSE)
  writeLines(as.character(valor), con = path, useBytes = TRUE)
}

ultimo_experimento <- leer_ultimo_experimento(CONFIG$experimento_state_file)
if (is.null(ultimo_experimento)) {
  ultimo_experimento <- as.integer(CONFIG$nro_experimento_inicial) - 1L
}

for (i in seq_along(CONFIG$semillas)) {
  ultimo_experimento <- ultimo_experimento + 1L
  semilla <- CONFIG$semillas[i]
  experimento <- ultimo_experimento
  run_dir <- file.path(CONFIG$output_dir, paste0("exp_", experimento, "_s_", semilla))

  cat("\n=== Corrida", i, "de", length(CONFIG$semillas), "===\n")
  cat("semilla:", semilla, "| experimento:", experimento, "| ratio:", CONFIG$ratio_canaritos, "| desvios:", CONFIG$desvios_canaritos, "| boruta:", CONFIG$aplicacion_boruta_flag, "\n")

  PARAM <- NULL

  ejecutar_corrida(
    semilla = semilla,
    experimento = experimento,
    ratio = CONFIG$ratio_canaritos,
    desvios = CONFIG$desvios_canaritos,
    boruta_flag = CONFIG$aplicacion_boruta_flag,
    base_path = base_path,
    work_dir = run_dir
  )

  guardar_ultimo_experimento(CONFIG$experimento_state_file, experimento)
}
